# Getting started with faro

This notebook runs a short timelapse on a virtual microscope and looks at the
results. You will meet the four objects that every faro experiment uses:

| Object | Job |
|--------|-----|
| **Microscope** | talks to the hardware (here: a simulation) |
| **Pipeline** | segments, tracks and measures cells on every frame |
| **Controller** | sends acquisition events to the microscope and frames to the pipeline |
| **Writer** | stores images and masks on disk |

Nothing here needs real hardware. Install faro with the virtual microscope:

```
uv sync --extra virtual-microscope
```

The [README](../README.md) explains every concept in more depth. Links in this
notebook point to the matching README section.

## 1. Connect to a microscope

The virtual microscope runs entirely in Python. `core` is a
[pymmcore-plus](https://github.com/pymmcore-plus/pymmcore-plus) core, so it
has the same API as a real Micro-Manager microscope. `mic` wraps the core in a
faro `Microscope`; this is the place where scope-specific behaviour lives.

If you have Micro-Manager installed you can use its demo devices instead:
`from faro.microscope.demo import MMDemo; mic = MMDemo()`.

In [1]:
from virtual_microscope.backends.optogenetic import setup_optogenetic
from faro.microscope.simulation import UniMMCoreSimulation
import faro.core.utils as utils

core, sim = setup_optogenetic(n_cells=20)
mic = UniMMCoreSimulation(mmc=core)
mic.init_scope()

utils.print_configs(core)

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 80, in emit
    self.doRollover()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 185, in doRollover
    self.rotate(self.baseFilename, dfn)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 121, in rotate
    os.rename(source, dest)
    ~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log' -> 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log.1'
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_ma

Config Groups
└── Channel
    ├── DAPI
    ├── membrane
    └── phase-contrast

## 2. Build the pipeline

The pipeline needs a segmentator, a tracker and a feature extractor. Here we
use built-in ones. Each segmentation gets a name (`"labels"`) so the other
components can refer to it. `storage_path` is where all results go.

See [Pipeline](../README.md#pipeline) in the README.

In [2]:
import tempfile

from faro.core.data_structures import SegmentationMethod
from faro.core.pipeline import ImageProcessingPipeline
from faro.feature_extraction.simple import SimpleFE
from faro.segmentation.base import OtsuSegmentator
from faro.tracking.trackpy import TrackerTrackpy

path = tempfile.mkdtemp(prefix="faro_getting_started_")

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=OtsuSegmentator(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(search_range=15),
)
print("results go to", path)

Directory C:\Users\lh21x018\AppData\Local\Temp\faro_getting_started_jvhc9fdy\tracks created 
results go to C:\Users\lh21x018\AppData\Local\Temp\faro_getting_started_jvhc9fdy


## 3. Check the segmentation on one frame

Snap a single image and run the segmentator on it. Do this before every
experiment: if the labels look wrong here, the tracks will be wrong too.

In [3]:
import matplotlib.pyplot as plt

core.setConfig("Channel", "phase-contrast")
core.snapImage()
test_img = core.getImage()
labels = OtsuSegmentator().segment(test_img)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(test_img, cmap="gray")
axes[0].set_title("Raw image")
axes[1].imshow(labels, cmap="nipy_spectral")
axes[1].set_title(f"Segmentation ({labels.max()} cells)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## 4. Define the experiment

`RTMSequence` describes what to acquire: how often, at which positions, in
which channels. Iterating over it gives one `RTMEvent` per timepoint and
position.

See [Experiment Definition](../README.md#experiment-definition) in the README.

In [4]:
from faro.core.data_structures import RTMSequence

seq = RTMSequence(
    time_plan={"interval": 1.0, "loops": 10},
    stage_positions=[{"x": 0.0, "y": 0.0, "z": 0.0}],
    channels=[{"config": "phase-contrast", "exposure": 50}],
)

events = list(seq)
print(f"{len(events)} events")
events[0]

10 events


RTMEvent(index={'t': 0, 'p': 0}, min_start_time=0.0, x_pos=0.0, y_pos=0.0, z_pos=0.0, channels=(Channel(config='phase-contrast', exposure=50.0, group='Channel'),))

## 5. Run the experiment

The controller connects microscope, pipeline and writer. `run_experiment`
returns immediately with a handle; `wait()` blocks until the last frame is
processed. Always call `finish_experiment()` at the end so every result is
flushed to disk.

See [Running](../README.md#running) in the README.

In [5]:
from faro.core.controller import Controller
from faro.core.writers import OmeZarrWriter

writer = OmeZarrWriter(storage_path=path)
ctrl = Controller(mic, pipeline, writer=writer)

handle = ctrl.run_experiment(events)
status = handle.wait()
ctrl.finish_experiment()
print(f"finished with state {status.state!r}, {status.n_frames_received} frames")

2026-09-07 15:16:14,514 INFO faro.core.controller: MDA run started: 10 events, stim_mode=current


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 80, in emit
    self.doRollover()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 185, in doRollover
    self.rotate(self.baseFilename, dfn)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 121, in rotate
    os.rename(source, dest)
    ~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log' -> 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log.1'
Call stack:
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpy

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 80, in emit
    self.doRollover()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 185, in doRollover
    self.rotate(self.baseFilename, dfn)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 121, in rotate
    os.rename(source, dest)
    ~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log' -> 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log.1'
Call stack:
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpy

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 80, in emit
    self.doRollover()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 185, in doRollover
    self.rotate(self.baseFilename, dfn)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 121, in rotate
    os.rename(source, dest)
    ~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log' -> 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log.1'
Call stack:
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpy

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 80, in emit
    self.doRollover()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 185, in doRollover
    self.rotate(self.baseFilename, dfn)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 121, in rotate
    os.rename(source, dest)
    ~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log' -> 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log.1'
Call stack:
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpy

...

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 80, in emit
    self.doRollover()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 185, in doRollover
    self.rotate(self.baseFilename, dfn)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 121, in rotate
    os.rename(source, dest)
    ~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log' -> 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log.1'
Call stack:
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpy

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 80, in emit
    self.doRollover()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 185, in doRollover
    self.rotate(self.baseFilename, dfn)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 121, in rotate
    os.rename(source, dest)
    ~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log' -> 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log.1'
Call stack:
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpy

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 80, in emit
    self.doRollover()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 185, in doRollover
    self.rotate(self.baseFilename, dfn)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 121, in rotate
    os.rename(source, dest)
    ~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log' -> 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log.1'
Call stack:
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpy

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 80, in emit
    self.doRollover()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 185, in doRollover
    self.rotate(self.baseFilename, dfn)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\logging\handlers.py", line 121, in rotate
    os.rename(source, dest)
    ~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log' -> 'C:\\Users\\lh21x018\\AppData\\Local\\pymmcore-plus\\pymmcore-plus\\logs\\pymmcore-plus.log.1'
Call stack:
  File "C:\Users\lh21x018\AppData\Roaming\uv\python\cpy

finished with state 'done', 10 frames


## 6. Look at the results

Tracks are parquet files, one per field of view. Each row is one cell in one
frame; `particle` is the track id, `label` the id in that frame's label image.

See [Storage](../README.md#storage) in the README for the full folder layout.

In [6]:
import os
import pandas as pd

tracks = pd.read_parquet(os.path.join(path, "tracks", "0_latest.parquet"))
print(f"{tracks['particle'].nunique()} tracks across {tracks['timestep'].nunique()} frames")
tracks.head()

17 tracks across 10 frames


,label,x,y,fov,timestep,fname,time,stim,channels,ref_channels,time_acquired,img_shape,particle,fov_timestep,area
0,1,4.587859,374.555911,0,0,000_00000,0.0,False,[phase-contrast],[],2026-09-07-15:16:14,"[512, 512]",0,0,313.0
1,2,31.557814,81.550191,0,0,000_00000,0.0,False,[phase-contrast],[],2026-09-07-15:16:14,"[512, 512]",1,0,787.0
2,3,36.844914,351.517623,0,0,000_00000,0.0,False,[phase-contrast],[],2026-09-07-15:16:14,"[512, 512]",2,0,993.0
3,4,102.259524,236.134524,0,0,000_00000,0.0,False,[phase-contrast],[],2026-09-07-15:16:14,"[512, 512]",3,0,840.0
4,5,107.605621,347.113327,0,0,000_00000,0.0,False,[phase-contrast],[],2026-09-07-15:16:14,"[512, 512]",4,0,1103.0


In [7]:
import zarr

store = zarr.open(os.path.join(path, "acquisition.ome.zarr"), mode="r")
raw = store["0"]          # (t, y, x); with several channels (t, c, y, x)
lbl = store["labels/labels/0"]
print("raw array shape:", raw.shape, " labels shape:", lbl.shape)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(raw[0], cmap="gray")
axes[0].set_title("Raw, first frame")
axes[1].imshow(raw[-1], cmap="gray")
axes[1].set_title("Raw, last frame")
axes[2].imshow(lbl[-1], cmap="nipy_spectral")
axes[2].set_title("Labels, last frame")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

raw array shape: (10, 512, 512)  labels shape: (10, 512, 512)


## Next steps

- [02_live_experiment.ipynb](02_live_experiment.ipynb) adds custom pipeline
  components, feedback stimulation and the napari GUI.
- [../templates/](../templates/) holds copy-and-fill notebooks for real experiments.